# Laboratorium 2: Problem mieszania — zaprojektuj tanią recepturę

[![Otwórz w Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdoHaha/automation_robotics_lab/blob/master/pl/lab_auto_2.ipynb)

Zrobotyzowany mieszalnik przygotowuje **100 kg przykładowej mieszanki karmy dla zwierząt**. Pięć składników różni się ceną i składem. Czy można spełnić wymagania jakościowe bez nadmiernych wydatków?

To klasyczny **liniowy problem mieszania**: wybieramy ilości składników, minimalizujemy koszt i spełniamy wymagania dotyczące średnich ważonych. Podobne modele stosuje się do stopów metali, paliw, nawozów i produkcji żywności.

W laboratorium 0 pojawiły się tabele w Pythonie, wykresy i elementy sterujące; w laboratorium 1 PuLP pomagał wybrać strukturę produkcji. Tutaj te same narzędzia służą do wyboru receptury. Przeanalizuj recepturę, jednostki i zapasy względem wymagań, a następnie zbadaj wrażliwość wyniku i dodaj własny element sterujący. Funkcje sprawdzające i rysujące ułatwiają eksperymentowanie z równaniami modelu i jego rozwiązaniami.

W tym ćwiczeniu:

- zamienisz udziały procentowe na ograniczenia liniowe z zachowaniem zgodności jednostek;
- zbudujesz i zinterpretujesz model minimalizacji w PuLP;
- sprawdzisz jakość receptury, ograniczenia aktywne i brak rozwiązania dopuszczalnego;
- zbadasz kompromis koszt–jakość za pomocą wykresów, suwaków i lokalnej analizy wrażliwości;
- uzasadnisz rekomendację receptury wynikami.

Uruchamiaj komórki od góry do dołu, wybierając **Środowisko wykonawcze → Uruchom wszystko** w Google Colab. Wszystkie dane i diagramy znajdują się w notatniku. Instalacja pakietów wymaga jednorazowego połączenia z internetem; eksperymenty nie wymagają zewnętrznych plików. Uproszczone składy i ceny służą nauce, a nie opracowaniu rzeczywistej karmy.

Nazwy w kodzie pozostają angielskie: `chicken` — kurczak, `beef` — wołowina, `eggs` — jaja, `milk` — mleko, `plant_filler` — wypełniacz roślinny; `protein` — białko, `fat` — tłuszcz, `moisture` — zawartość wody. Pozwala to używać tych samych nazw w tabelach, indeksowaniu i kolejnych laboratoriach. `Optimal` oznacza znalezione rozwiązanie optymalne, a `Infeasible` — brak rozwiązania dopuszczalnego.

In [ ]:
# Zainstaluj biblioteki w środowisku notatnika, w tym używaną w kursie wersję PuLP.
%pip install -q 'pulp>=3.3,<4' numpy pandas matplotlib ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import display, clear_output
import ipywidgets as widgets
import pulp

# Colab wymaga tego do wyświetlania widżetów; zwykły Jupyter nie.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False})
SOLVER = pulp.PULP_CBC_CMD(msg=False)
print("PuLP", pulp.__version__, "| CBC dostępny:", bool(SOLVER.available()))
if not SOLVER.available():
    raise RuntimeError("Brak CBC. Uruchom komórkę instalacji w świeżym środowisku Colab.")

## 1. Przeczytaj opis receptury

Zespół zakupowy dostarcza pięć składników. Mieszalnik może odmierzać ułamkowe ilości kilogramów, więc ilości są **zmiennymi ciągłymi**. Masa receptury musi być dokładnie równa zadanej masie partii.

Każdy wiersz poniżej opisuje **jeden kilogram** składnika. Na przykład `protein = 0.38` oznacza 0,38 kg białka na kg kurczaka, czyli 38% masy. Pozostałą masę mogą stanowić węglowodany, popiół i inne substancje; podane udziały nie muszą sumować się do jedności.

Granice udziałów odzwierciedlają praktyczne reguły receptury: co najmniej 10% kurczaka i 5% wypełniacza roślinnego oraz nie więcej niż podane udziały maksymalne. Są to ograniczenia proporcji, a nie zapasu magazynowego. Kolumny `cost_pln_per_kg`, `min_share` i `max_share` oznaczają odpowiednio cenę w PLN/kg oraz minimalny i maksymalny udział masowy.

In [ ]:
# Umieść ceny i składy na kg oraz granice udziałów w notatniku, aby odtwarzać eksperymenty.
ingredients = pd.DataFrame([
    # nazwa,      PLN/kg, białko, tłuszcz, woda, udział min., udział maks.
    ("chicken",      6.95,   0.380, 0.090, 0.46, 0.10, 0.60),
    ("beef",         9.60,   0.340, 0.170, 0.40, 0.00, 0.45),
    ("eggs",         5.40,   0.130, 0.110, 0.72, 0.00, 0.25),
    ("milk",         1.15,   0.034, 0.033, 0.88, 0.00, 0.20),
    ("plant_filler", 1.05,   0.160, 0.030, 0.10, 0.05, 0.35),
], columns=["ingredient", "cost_pln_per_kg", "protein", "fat", "moisture",
            "min_share", "max_share"]).set_index("ingredient")

# Zachowaj kolory składników w diagramie, wykresach receptury i seriach badań jakości.
COLORS = dict(zip(ingredients.index,
                  ["#4393c3", "#b35806", "#e8b923", "#8073ac", "#49a06c"]))
display(ingredients)

In [ ]:
# Narysuj składniki, strzałki i wynik mieszania bez zewnętrznych plików graficznych.
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set(xlim=(0, 11), ylim=(-0.8, 4.8))
ax.axis("off")
for y, (name, row) in enumerate(ingredients.iterrows()):
    box = FancyBboxPatch((0.1, y - 0.33), 3.5, 0.66, boxstyle="round,pad=0.05",
                         facecolor=COLORS[name], edgecolor="white", alpha=0.25)
    ax.add_patch(box)
    ax.text(0.25, y, f"{name.replace('_', ' ').title()}  |  {row.cost_pln_per_kg:.2f} PLN/kg"
            f"\nBiałko {row.protein:.1%}; tłuszcz {row.fat:.1%}", va="center", fontsize=10)
    ax.annotate("", xy=(5, 2), xytext=(3.7, y),
                arrowprops={"arrowstyle": "->", "color": COLORS[name], "lw": 1.8})
ax.add_patch(FancyBboxPatch((5, 1.25), 2.2, 1.5, boxstyle="round,pad=0.1",
                           facecolor="#e6edf3", edgecolor="#345"))
ax.text(6.1, 2, "ZROBOTYZOWANY\nMIESZALNIK\nWybierz kg każdego\nskładnika", ha="center", va="center")
ax.annotate("", xy=(8.1, 2), xytext=(7.25, 2), arrowprops={"arrowstyle": "->", "lw": 2})
ax.text(9.4, 2, "Partia 100 kg\nBiałko ≥ 24%\nTłuszcz ≥ 8%\nWoda ≤ 60%", ha="center", va="center",
        bbox={"boxstyle": "round", "facecolor": "#edf7ed", "edgecolor": "#486"})
ax.set_title("Składy wejściowe → wybór receptury → wymagania jakościowe", fontsize=14)
plt.show()

### Przewidź wynik przed uruchomieniem

1. Który składnik jest najtańszy? Dlaczego receptura złożona tylko z niego może nie spełniać wymagań?
2. Czy składnik o największej zawartości białka jest zawsze najlepszym zakupem? Porównaj **koszt jednego kg białka**, a następnie wyjaśnij, dlaczego sam ten wskaźnik nie wystarcza do wyboru całej receptury.
3. Jeżeli masa partii wzrośnie dwukrotnie, a wszystkie reguły są procentowe, czy udziały składników powinny się zmienić?

## 2. Przykład wprowadzający: tylko kurczak i wypełniacz roślinny

Na chwilę pomiń tłuszcz, wodę i granice udziałów. Przygotuj 100 kg mieszanki o zawartości białka co najmniej 24%. Niech $c$ oznacza kg kurczaka, a $f$ — kg wypełniacza:

$$\min 6.95c + 1.05f,\qquad c+f=100,\qquad 0.38c+0.16f\geq24,\qquad c,f\geq0.$$

Ręcznie podstaw $f=100-c$. Jaka jest najmniejsza dopuszczalna wartość $c$? Ponieważ kurczak jest droższy, receptura na tej granicy jest najtańszym rozwiązaniem dopuszczalnym. Przewidź wynik przed uruchomieniem wersji w PuLP.

In [ ]:
# Zbuduj model kosztu dwóch składników, aby pokazać każde równanie przed uogólnieniem.
warmup = pulp.LpProblem("Two_ingredient_recipe", pulp.LpMinimize)
chicken_kg = pulp.LpVariable("chicken_kg", lowBound=0)
filler_kg = pulp.LpVariable("filler_kg", lowBound=0)
warmup += 6.95 * chicken_kg + 1.05 * filler_kg, "Cost_PLN"
warmup += chicken_kg + filler_kg == 100, "Batch_kg"
warmup += 0.38 * chicken_kg + 0.16 * filler_kg >= 24, "Protein_kg"
# Sprawdź status rozwiązania przed rysowaniem lub odczytem receptury.
warmup.solve(SOLVER)
warmup_status = pulp.LpStatus[warmup.status]
print("Status:", warmup_status)
if warmup_status == "Optimal":
    print(f"Kurczak: {chicken_kg.value():.2f} kg; wypełniacz: {filler_kg.value():.2f} kg")
    print(f"Koszt: {pulp.value(warmup.objective):.2f} PLN")
    # Przy filler = 100 - chicken wspólna siatka pokazuje jakość i koszt.
    grid = np.linspace(0, 100, 301)
    protein_percent = 0.38 * grid + 0.16 * (100 - grid)
    costs = 6.95 * grid + 1.05 * (100 - grid)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(grid, protein_percent)
    axes[0].axhline(24, color="#ba332b", linestyle="--", label="wymaganie 24%")
    axes[0].fill_between(grid, 0, 40, where=protein_percent >= 24, color="#68ab75", alpha=0.15)
    axes[0].set(xlabel="Kurczak w partii 100 kg (kg)", ylabel="Białko (%)", ylim=(0, 40))
    axes[0].legend()
    axes[1].plot(grid, costs)
    axes[1].scatter([chicken_kg.value()], [pulp.value(warmup.objective)], color="#ba332b",
                    zorder=3, label="Najtańsza receptura dopuszczalna")
    axes[1].set(xlabel="Kurczak w partii 100 kg (kg)", ylabel="Koszt partii (PLN)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 3. Od dwóch składników do pełnego modelu liniowego

Niech $x_i$ oznacza **kg składnika $i$**, a $B=100$ kg — stałą masę partii. Dla każdego składnika $c_i$ to cena w PLN/kg, $p_i$, $f_i$ i $m_i$ to udziały masowe substancji, a $l_i$, $u_i$ — granice udziału składnika.

$$\begin{aligned}
\min\quad &\sum_i c_i x_i &&\text{koszt w PLN}\\
\text{przy ograniczeniach}\quad &\sum_i x_i=B &&\text{masa partii w kg}\\
&\sum_i p_i x_i\geq 0.24B &&\text{co najmniej 24 kg białka na 100 kg}\\
&\sum_i f_i x_i\geq 0.08B &&\text{co najmniej 8 kg tłuszczu na 100 kg}\\
&\sum_i m_i x_i\leq 0.60B &&\text{co najwyżej 60 kg wody na 100 kg}\\
&l_i B\leq x_i\leq u_i B &&\text{granice udziału składnika.}
\end{aligned}$$

**Dlaczego model jest liniowy?** Średnia ważona zawartości białka wynosi $\sum_i p_i x_i/B$. Ponieważ $B$ jest dodatnim, stałym parametrem, mnożymy obie strony wymagania procentowego przez $B$. Wszystkie współczynniki są stałe; nigdy nie mnożymy dwóch zmiennych decyzyjnych. Model oparty na udziałach $s_i=x_i/B$ także jest liniowy i ma ograniczenie $\sum_i s_i=1$.

**Sprawdzenie jednostek:** `(kg białka / kg składnika) × kg składnika = kg białka`. Wpisanie `24` zamiast `0.24` po prawej stronie wymagałoby 2400 kg białka w partii o masie 100 kg.

Najpierw przeczytaj model zapisany bez funkcji pomocniczej. Dopasuj każdą linię kodu do odpowiedniej linii równań.

In [ ]:
# Zmienne opisują kg składników; stała masa partii przelicza procenty na wymagane kg.
batch_kg = 100
full_model = pulp.LpProblem("Five_ingredient_recipe", pulp.LpMinimize)
x = {name: pulp.LpVariable(f"kg_{name}", lowBound=0) for name in ingredients.index}
full_model += pulp.lpSum(ingredients.loc[i, "cost_pln_per_kg"] * x[i] for i in x), "Cost_PLN"
full_model += pulp.lpSum(x.values()) == batch_kg, "Batch_kg"
full_model += pulp.lpSum(ingredients.loc[i, "protein"] * x[i] for i in x) >= 0.24 * batch_kg, "Protein_kg"
full_model += pulp.lpSum(ingredients.loc[i, "fat"] * x[i] for i in x) >= 0.08 * batch_kg, "Fat_kg"
full_model += pulp.lpSum(ingredients.loc[i, "moisture"] * x[i] for i in x) <= 0.60 * batch_kg, "Moisture_kg"
for i in x:
    full_model += x[i] >= ingredients.loc[i, "min_share"] * batch_kg, f"Min_{i}"
    full_model += x[i] <= ingredients.loc[i, "max_share"] * batch_kg, f"Max_{i}"
# Odczytaj ilości dopiero po znalezieniu optimum pełnego modelu.
full_model.solve(SOLVER)
print("Status:", pulp.LpStatus[full_model.status])
if full_model.status == pulp.LpStatusOptimal:
    display(pd.Series({i: x[i].value() for i in x}, name="Ilość (kg)").to_frame())
    print(f"Minimalny koszt: {pulp.value(full_model.objective):.2f} PLN")

### Zadanie 1 — obejrzyj i zmień model

- **Przewidź:** Które ograniczenia powinny być aktywne, gdy model unika drogich składników?
- **Uruchom:** Porównaj obliczoną recepturę ze swoim przewidywaniem.
- **Zmień:** W modelu zapisanym bez funkcji podnieś wymaganie białka z `0.24` do `0.28`. Uruchom tę komórkę ponownie i porównaj koszt oraz ilości. Następnie przywróć `0.24`.
- **Wyjaśnij:** Dlaczego minimalne wymaganie używa `>=`, a ograniczenie wody — `<=`? Dlaczego zaokrąglenie każdej ilości do pełnych kg może naruszyć wymagania?

## 4. Solver wielokrotnego użytku ze sprawdzaniem wyniku

Budowanie nowego modelu przy każdym eksperymencie zapobiega przypadkowemu gromadzeniu starych ograniczeń. Poniższe funkcje oddzielają prostą granicę diagnostyczną od kompletnego modelu optymalizacyjnego.

### Funkcja pomocnicza: optymistyczna granica zawartości białka

`maximum_protein_from_share_limits(data)` przyjmuje tabelę składników z udziałami białka oraz dolnymi i górnymi granicami udziału składników. Rezerwuje udziały minimalne, a następnie przydziela pozostały udział składnikom o największej zawartości białka, do ich limitów. Zwraca jeden **ułamek określający udział białka**, a nie recepturę; niczego nie wyświetla i nie wywołuje solvera. Pomija tłuszcz i wodę, więc może wykazać niemożliwość żądania, ale nie dowodzi dopuszczalności pełnej receptury. Użycie tej samej funkcji w diagnostyce nieudanego rozwiązania i w samodzielnym eksperymencie zapewnia spójne wyjaśnienie. Minimalne udziały muszą mieścić się w jednej partii, a maksymalne — umożliwiać jej wypełnienie.

In [ ]:
def maximum_protein_from_share_limits(data):
    """Zwróć optymistyczną maksymalną zawartość białka przy granicach udziałów.
    
    data jest tabelą DataFrame indeksowaną składnikami, z protein (kg białka/kg
    składnika), min_share i max_share jako ułamkami. Suma minimów nie może
    przekraczać jedności, suma maksimów musi wynosić co najmniej jeden, a każde
    minimum nie może przekraczać maksimum. Brak argumentów domyślnych.
    Zarezerwuj minima i wypełnij resztę według malejącej zawartości białka.
    Zwróć ułamek float, pomijając tłuszcz i wodę; bez wyświetlania, solvera
    ani globalnej tabeli składników. Dane wejściowe nie są modyfikowane."""
    # Zarezerwuj wymagane składniki przed przydzieleniem pozostałego udziału partii.
    shares = data["min_share"].copy()
    remaining = 1.0 - shares.sum()
    # Przydział zachłanny jest poprawny dla tej granicy jednej cechy, a nie dla pełnego LP mieszania.
    for name in data.sort_values("protein", ascending=False).index:
        addition = min(remaining, data.loc[name, "max_share"] - shares[name])
        shares[name] += addition
        remaining -= addition
    return float((shares * data["protein"]).sum())

### Funkcja pomocnicza: rozwiąż i sprawdź jedną recepturę

`solve_blend(batch_kg=100, min_protein=0.24, min_fat=0.08, max_moisture=0.60,
chicken_cost_multiplier=1.0, data=None)` przyjmuje masę partii w kg, trzy granice jakości jako ułamki, bezwymiarowy mnożnik ceny kurczaka (cena bazowa w PLN/kg) i opcjonalną tabelę składników. `data=None` kopiuje wspólną tabelę `ingredients`; przekazana własna tabela także jest kopiowana, więc eksperymenty nie zmieniają danych bazowych. Na przykład `solve_blend(batch_kg=200, min_protein=0.28)` zmienia tylko masę partii i wymaganie białka.

Funkcja sprawdza dane, buduje te same równania kosztu i jakości co powyżej, rozwiązuje model wspólnym solverem `SOLVER` i sprawdza status. Dla optimum ponownie oblicza masę, jakość i zapasy względem wymagań, po czym zwraca słownik z `recipe` (recepturą), `quality` (jakością), `constraints` (ograniczeniami), `quantities_kg` (ilościami) i `cost_pln` (kosztem), a także model i status. Niczego nie wyświetla. Tabele, wykresy, serie eksperymentów i elementy sterujące korzystają z tej samej sprawdzonej struktury.

**Najpierw sprawdź status.** Gdy nie znaleziono optimum, wynik zawiera diagnostykę, a pola rozwiązania mają wartość `None`. Funkcja kończy działanie przed odczytem zmiennych decyzyjnych, ponieważ wartości pozostawione przez solver po stwierdzeniu braku rozwiązania dopuszczalnego nie opisują użytecznej receptury. Optymistyczna granica białka wskazuje jedną możliwą przyczynę; wzajemne oddziaływanie ograniczeń jakości może wymagać dokładniejszej analizy.

In [ ]:
def solve_blend(batch_kg=100, min_protein=0.24, min_fat=0.08,
                max_moisture=0.60, chicken_cost_multiplier=1.0, data=None):
    """Rozwiąż i sprawdź ciągły liniowy problem mieszania, bez wyświetlania.
    
    batch_kg=100 to dodatnia masa w kg; min_protein=0.24, min_fat=0.08 oraz
    max_moisture=0.60 to udziały masowe. chicken_cost_multiplier=1.0 skaluje
    wyłącznie cenę kurczaka w PLN/kg. data=None kopiuje globalną ingredients;
    opcjonalny indeksowany DataFrame podaje ceny, składy i granice udziałów.
    Korzysta z globalnego SOLVER i maximum_protein_from_share_limits w diagnostyce.
    
    Zwraca słownik ze status, model, batch_kg i komunikatami diagnostics.
    Optimum zawiera także cost_pln, quantities_kg (Series), recipe (DataFrame
    z kg, share_pct i kosztami składników), quality (osiągnięte/wymagane udziały,
    zapasy i sprawdzenia) oraz constraints (osiągnięte/graniczne kg, jednostki,
    luz/nadwyżkę, aktywność i ceny dualne LP). Pola rozwiązania pozostają None,
    jeżeli status nie jest Optimal. Kopiuje dane; błędne dane wywołują ValueError."""
    # Pracuj na kopii, aby zmiana ceny nie wpływała na dane innych eksperymentów.
    table = ingredients.copy(deep=True) if data is None else data.copy(deep=True)
    # Sprawdź jednostki, ułamki, ceny i granice udziałów przed budowaniem modelu.
    parameters = [batch_kg, min_protein, min_fat, max_moisture, chicken_cost_multiplier]
    if not np.isfinite(parameters).all() or batch_kg <= 0 or chicken_cost_multiplier < 0:
        raise ValueError("Masa partii musi być dodatnia i skończona, a mnożnik ceny — nieujemny i skończony.")
    if not all(0 <= v <= 1 for v in [min_protein, min_fat, max_moisture]):
        raise ValueError("Granice składników odżywczych muszą być ułamkami od 0 do 1.")
    required = ["cost_pln_per_kg", "protein", "fat", "moisture", "min_share", "max_share"]
    if table.empty or not table.index.is_unique or "chicken" not in table.index:
        raise ValueError("Nazwy składników muszą być unikalne i zawierać chicken.")
    if not np.isfinite(table[required].to_numpy()).all():
        raise ValueError("Dane składników muszą zawierać skończone wartości liczbowe.")
    fractions = table[["protein", "fat", "moisture", "min_share", "max_share"]]
    if ((fractions < 0) | (fractions > 1)).any().any():
        raise ValueError("Skład i udziały muszą być ułamkami z przedziału [0, 1].")
    if (table["cost_pln_per_kg"] < 0).any() or (table.min_share > table.max_share).any():
        raise ValueError("Koszty muszą być nieujemne; udziały minimalne nie mogą przekraczać maksymalnych.")
    # Zbuduj nowy model: koszt jako funkcja celu, bilans masy, jakość i granice udziałów.
    table.loc["chicken", "cost_pln_per_kg"] *= chicken_cost_multiplier
    model = pulp.LpProblem("Blend", pulp.LpMinimize)
    variables = {i: pulp.LpVariable(f"kg_{i}", lowBound=0) for i in table.index}
    model += pulp.lpSum(table.loc[i, "cost_pln_per_kg"] * variables[i] for i in table.index), "Cost_PLN"
    model += pulp.lpSum(variables.values()) == batch_kg, "Batch_kg"
    model += pulp.lpSum(table.loc[i, "protein"] * variables[i] for i in table.index) >= min_protein * batch_kg, "Protein_kg"
    model += pulp.lpSum(table.loc[i, "fat"] * variables[i] for i in table.index) >= min_fat * batch_kg, "Fat_kg"
    model += pulp.lpSum(table.loc[i, "moisture"] * variables[i] for i in table.index) <= max_moisture * batch_kg, "Moisture_kg"
    for i in table.index:
        model += variables[i] >= table.loc[i, "min_share"] * batch_kg, f"Min_{i}"
        model += variables[i] <= table.loc[i, "max_share"] * batch_kg, f"Max_{i}"
    # Najpierw zapisz status; nieudane rozwiązanie nie może udostępniać pozostałych wartości zmiennych.
    model.solve(SOLVER)
    result = {"status": pulp.LpStatus[model.status], "model": model,
              "batch_kg": batch_kg, "cost_pln": None, "quantities_kg": None,
              "recipe": None, "quality": None, "constraints": None, "diagnostics": []}
    if model.status != pulp.LpStatusOptimal:
        # Wyjaśnij naruszenia prostych granic koniecznych; nie obejmują wszystkich konfliktów.
        if table.min_share.sum() > 1 + 1e-9:
            result["diagnostics"].append("Suma minimalnych udziałów składników przekracza 100%.")
        elif table.max_share.sum() < 1 - 1e-9:
            result["diagnostics"].append("Maksymalne udziały składników nie pozwalają wypełnić całej partii.")
        else:
            upper = maximum_protein_from_share_limits(table)
            if min_protein > upper + 1e-9:
                result["diagnostics"].append(
                    f"Wymagana zawartość białka {min_protein:.1%} przekracza górną granicę wynikającą z udziałów {upper:.1%}.")
        if min_fat > table.fat.max() + 1e-9:
            result["diagnostics"].append("Wymagana zawartość tłuszczu przekracza zawartość w najtłustszym składniku.")
        if max_moisture < table.moisture.min() - 1e-9:
            result["diagnostics"].append("Dopuszczalna zawartość wody jest mniejsza niż w najsuchszym składniku.")
        if not result["diagnostics"]:
            result["diagnostics"].append(
                "Prosta granica nie wyjaśnia statusu. Zbadaj współdziałające ograniczenia jakości i udziałów; poluzuj jedno naraz.")
        return result

    # Odczytaj tylko optimum i przelicz udziały oraz koszty w pierwotnych jednostkach.
    quantities = pd.Series({i: variables[i].value() for i in table.index}, name="kg")
    recipe = table.copy()
    recipe["kg"] = quantities
    recipe["share_pct"] = 100 * quantities / batch_kg
    recipe["ingredient_cost_pln"] = quantities * table.cost_pln_per_kg
    targets = {"protein": (min_protein, ">="), "fat": (min_fat, ">="),
               "moisture": (max_moisture, "<=")}
    # Przelicz sumy ważone na procenty i zapasy jakości z właściwym znakiem.
    quality_rows = []
    for nutrient, (target, sense) in targets.items():
        actual = float((quantities * table[nutrient]).sum() / batch_kg)
        margin = actual - target if sense == ">=" else target - actual
        quality_rows.append({"nutrient": nutrient, "actual_pct": 100 * actual,
                             "target_pct": 100 * target, "rule": sense,
                             "margin_percentage_points": 100 * margin,
                             "meets_target": margin >= -1e-7})

    # Opisz każde ograniczenie wartością osiągniętą, granicą, zapasem i lokalną ceną dualną.
    checks = []
    for name, constraint in model.constraints.items():
        # PuLP zapisuje (lewa − prawa); luz lub nadwyżka z właściwym znakiem są nieujemne dla rozwiązania dopuszczalnego.
        residual = float(pulp.value(constraint))
        slack = (-residual if constraint.sense == pulp.LpConstraintLE else
                 residual if constraint.sense == pulp.LpConstraintGE else abs(residual))
        units = {"Protein_kg": "kg białka", "Fat_kg": "kg tłuszczu",
                 "Moisture_kg": "kg wody"}.get(name, "kg składnika/partii")
        checks.append({"constraint": name, "sense": {0: "=", 1: ">=", -1: "<="}[constraint.sense],
                       "achieved_kg": residual - constraint.constant,
                       "bound_kg": -constraint.constant, "units": units,
                       "slack_or_surplus_kg": slack, "binding": abs(slack) <= 1e-5,
                       "dual_pln_per_rhs_kg": constraint.pi})
    quality = pd.DataFrame(quality_rows).set_index("nutrient")
    constraint_table = pd.DataFrame(checks).set_index("constraint")
    # Niezależnie sprawdź rachunki przed przedstawieniem rozwiązania.
    assert abs(quantities.sum() - batch_kg) <= 1e-5 * max(1, batch_kg)
    assert quality.meets_target.all(), "Zwrócona receptura nie spełnia wymagania jakościowego."
    assert (constraint_table.slack_or_surplus_kg >= -1e-5 * max(1, batch_kg)).all()
    result.update(cost_pln=float(pulp.value(model.objective)), quantities_kg=quantities,
                  recipe=recipe, quality=quality, constraints=constraint_table)
    return result


# Zachowaj wynik domyślny, aby bazowe tabele i wykresy opisywały tę samą recepturę.
baseline = solve_blend()
print("Status wariantu bazowego:", baseline["status"])
if baseline["status"] == "Optimal":
    display(baseline["recipe"][["kg", "share_pct", "ingredient_cost_pln"]].round(3))
    display(baseline["quality"].round(4))
    print(f"Koszt = {baseline['cost_pln']:.2f} PLN; {baseline['cost_pln'] / 100:.3f} PLN/kg")

## 5. Zobacz rozwiązanie i zapasy względem wymagań

Wykres receptury odpowiada na pytanie **co trafia do mieszalnika**. Wykres jakości pokazuje **czy mieszanka spełnia wymagania**. Linia minimum powinna leżeć na wysokości słupka lub poniżej niego; linia maksimum — na wysokości słupka lub powyżej niego.

**Ograniczenie aktywne** jest spełnione na granicy. Dla reguły `<=` **luz = granica − wartość osiągnięta**, a dla `>=` **nadwyżka = wartość osiągnięta − granica**. W rozwiązaniu dopuszczalnym oba zapasy są nieujemne. Tabela podaje wartości osiągnięte, granice, jednostki i `slack_or_surplus_kg` (luz lub nadwyżkę w kg). Nadwyżka białka to dodatkowe kg białka ponad wymaganie; luz w ograniczeniu wody to niewykorzystana część dopuszczalnej ilości wody.

Tabela jakości przedstawia te same zapasy w **punktach procentowych**. Dla równości kolumna zawiera wartość bezwzględną reszty numerycznej: powinna być bliska zeru i nie stanowi zasobu do wykorzystania. Ograniczenie aktywne ma zerowy zapas z dokładnością do tolerancji numerycznej. Cena dualna odpowiada na inne pytanie: jak zmieni się koszt po przesunięciu granicy.

`plot_blend(result)` przyjmuje słownik zwrócony przez `solve_blend`. Sprawdza status przed odczytem receptury, wypisuje koszt partii, rysuje ilości składników w kg i procentowy skład, a następnie wyświetla tabelę jakości. Przy braku optimum wypisuje diagnostykę. Zwraca `None` i nie rozwiązuje modelu ponownie. Wspólny słownik `COLORS` zapewnia spójny kolor każdego składnika; nowe nazwy otrzymują kolor szary. Użycie tej samej funkcji dla przypadku bazowego, nazwanych eksperymentów i obsługi suwaków ułatwia porównanie statusów i jakości.

Angielskie nagłówki wyników pozostają nazwami używanymi w kodzie: `kg` — masa, `share_pct` — udział procentowy, `ingredient_cost_pln` — koszt składnika, `actual_pct` — udział osiągnięty, `target_pct` — udział wymagany, `margin_percentage_points` — zapas w punktach procentowych, `meets_target` — spełnienie wymagania. W tabeli ograniczeń `achieved_kg` oznacza wartość osiągniętą, `bound_kg` — granicę, `binding` — aktywność, a `dual_pln_per_rhs_kg` — cenę dualną w PLN na dodatkowy kg prawej strony ograniczenia.

In [ ]:
def plot_blend(result):
    """Wyświetl wykres receptury i jakości wyniku solve_blend; zwróć None.
    
    result to słownik solvera ze status/diagnostics, a przy statusie Optimal także
    batch_kg, cost_pln, recipe (kg i share_pct) oraz quality (osiągnięte/wymagane
    procenty i reguły >=/<=). Status inny niż Optimal wyświetla diagnostykę
    bez odczytu rozwiązania. Optimum wyświetla koszt, dwa panele Matplotlib
    i tabelę jakości DataFrame. Korzysta z globalnych COLORS, np, plt i display;
    nie zmienia danych ani nie rozwiązuje modelu ponownie."""
    # Sprawdź status przed odczytem pól liczbowych receptury.
    if result["status"] != "Optimal":
        print("Status:", result["status"], "— brak receptury do narysowania.")
        for message in result["diagnostics"]:
            print(" •", message)
        return
    # Rysuj wyłącznie na podstawie sprawdzonej receptury i tabeli jakości.
    recipe, quality = result["recipe"], result["quality"]
    print(f"Optymalna partia {result['batch_kg']:g} kg: {result['cost_pln']:.2f} PLN"
          f" ({result['cost_pln'] / result['batch_kg']:.3f} PLN/kg)")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    # Podpisz masy udziałami, aby pokazać oba sposoby opisu ilości.
    axes[0].barh(recipe.index, recipe.kg, color=[COLORS.get(i, "#777") for i in recipe.index])
    for j, (_, row) in enumerate(recipe.iterrows()):
        axes[0].text(row.kg + result["batch_kg"] * 0.012, j,
                     f"{row.kg:.1f} kg / {row.share_pct:.1f}%", va="center", fontsize=9)
    axes[0].set(xlabel="Masa składnika (kg)", title="Receptura", xlim=(0, result["batch_kg"] * 0.85))
    # Porównaj osiągnięty procent każdej cechy z linią minimum lub maksimum.
    positions = np.arange(len(quality))
    axes[1].bar(positions, quality.actual_pct, color="#4393c3", alpha=0.7, label="Osiągnięto")
    for pos, (_, row) in zip(positions, quality.iterrows()):
        axes[1].plot([pos - 0.38, pos + 0.38], [row.target_pct] * 2, color="#ba332b", lw=2)
        axes[1].text(pos, max(row.actual_pct, row.target_pct) + 2,
                     f"{row.rule} {row.target_pct:g}%", ha="center", fontsize=9)
    axes[1].plot([], [], color="#ba332b", lw=2, label="Wymaganie")
    axes[1].set(xticks=positions, xticklabels=quality.index,
                ylabel="Skład partii (%)", title="Kontrola jakości", ylim=(0, 100))
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    # Wyświetl pola tabeli obok kontroli wizualnej, zaokrąglone dla czytelności.
    display(quality.round(4))


plot_blend(baseline)
if baseline["status"] == "Optimal":
    display(baseline["constraints"].round(5))

### Samodzielnie zbuduj wykres

Wykres składa się z **figury** — całego obszaru rysunku — oraz obiektu **axes**, czyli pojedynczego pola wykresu. `ax.barh(labels, values)` rysuje poziomy słupek dla każdej wartości; `ax.set(...)` ustawia opisy osi. Poniżej znajduje się krótki, działający wykres, który możesz zmienić bez czytania całej funkcji rysującej.

In [ ]:
# Narysuj sprawdzone koszty składników w partii, a nie ceny jednostkowe czy niesprawdzone zmienne.
if baseline["status"] == "Optimal":
    fig, ax = plt.subplots(figsize=(7, 3))  # utwórz jedną figurę i jedno pole wykresu
    ax.barh(baseline["recipe"].index, baseline["recipe"].ingredient_cost_pln,
            color=[COLORS[i] for i in baseline["recipe"].index])
    ax.set(xlabel="Koszt składnika w partii (PLN)", title="Z czego wynika rachunek?")
    plt.tight_layout()  # zostaw miejsce na opisy
    plt.show()

**Zadanie z wykresem:** zamiast kosztów składników pokaż ich udziały procentowe, a następnie zmień opis osi i tytuł. Dodaj pionową linię przez `ax.axvline(...)` jako sensowny punkt odniesienia dla udziału składnika. Wyjaśnij, dlaczego jest to punkt odniesienia, a nie uniwersalne ograniczenie receptury.

### Zadanie 2 — wyjaśnij recepturę

1. Które składniki osiągają minimalny lub maksymalny udział? Czy pominięcie składnika zawsze oznacza jego niską jakość?
2. Które ograniczenia jakości są aktywne? Wskaż tę samą granicę w tabeli i na wykresie.
3. **Na papierze** dodaj 1 kg najtańszego składnika i usuń 1 kg kurczaka. Przelicz zawartość białka i tłuszczu. Czy zmieniona mieszanka nadal jest dopuszczalna?
4. Porównaj recepturę z przykładem dwóch składników. Wyjaśnij, dlaczego tam koszt był niższy: rozwiązywano model o innym zestawie ograniczeń.

## 6. Ostrzejsze wymaganie może spowodować brak rozwiązania dopuszczalnego

Największy udział białka w pojedynczym składniku wynosi 38%, ale granice udziałów uniemożliwiają użycie samego kurczaka. Aby wyznaczyć konieczną górną granicę białka, zacznij od minimalnych udziałów i wypełnij pozostałą część partii składnikami o najwyższej zawartości białka, do ich limitów. Pomijasz inne wymagania jakościowe, więc jest to **optymistyczna granica górna**, a nie gwarancja dopuszczalnej receptury.

Przewidź, czy da się osiągnąć **36% białka**. Następnie uruchom poniższy scenariusz. Brak rozwiązania dopuszczalnego jest oczekiwanym wynikiem eksperymentu: nie istnieje receptura spełniająca wszystkie reguły. To coś innego niż wyjątek Pythona lub brak solvera.

In [ ]:
# Porównaj konieczną granicę wynikającą z udziałów z celowo niemożliwym wymaganiem jakości.
protein_upper_bound = maximum_protein_from_share_limits(ingredients)
print(f"Górna granica białka wynikająca z udziałów: {protein_upper_bound:.1%}")
impossible = solve_blend(min_protein=0.36)
plot_blend(impossible)
assert impossible["status"] == "Infeasible"
assert impossible["quantities_kg"] is None

### Zadanie 3 — zdiagnozuj problem i zaproponuj naprawę

- Które założenie zmienisz: wymaganie białka, maksymalny udział kurczaka czy listę składników?
- Zmień jedną rzecz w `ingredients.copy()` i przekaż kopię funkcji jako `data=...`. Nie usuwaj tabeli bazowej.
- Rozwiąż model ponownie i sprawdź **wszystkie** wymagania jakościowe. Górna granica białka większa od wymaganego udziału nie dowodzi dopuszczalności, gdy obowiązują też reguły tłuszczu i wody.
- Wyjaśnij, dlaczego zwiększenie partii ze 100 kg do 200 kg nie naprawi sprzecznych reguł procentowych.

## 7. Kompromis koszt–jakość: zbadaj serię wariantów przed wyborem

Kierownictwo proponuje wyższą deklarowaną zawartość białka. Każdy punkt poniżej to **nowa optymalizacja**, a nie stara receptura z nową etykietą. Przewidź kształt wykresu: czy minimalny koszt może spaść, gdy minimalne wymaganie białka rośnie, a pozostałe dane się nie zmieniają?

In [ ]:
# Optymalizuj każdy wariant; brak rozwiązania dopuszczalnego zachowuje status, ale nie koszt liczbowy.
sweep_rows = []
for target in np.linspace(0.14, 0.38, 25):
    result = solve_blend(min_protein=float(target))
    row = {"protein_target_pct": 100 * target, "status": result["status"],
           "cost_pln_per_kg": np.nan}
    if result["status"] == "Optimal":
        row["cost_pln_per_kg"] = result["cost_pln"] / result["batch_kg"]
        row.update({f"share_{i}": result["recipe"].loc[i, "share_pct"] for i in ingredients.index})
    sweep_rows.append(row)
sweep = pd.DataFrame(sweep_rows)
# Oddziel receptury dopuszczalne od nieudanych rozwiązań przed rysowaniem kosztów i udziałów.
feasible = sweep[sweep.status == "Optimal"]
infeasible = sweep[sweep.status != "Optimal"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(feasible.protein_target_pct, feasible.cost_pln_per_kg, "o-", color="#345f93")
if not infeasible.empty:
    axes[0].scatter(infeasible.protein_target_pct, np.zeros(len(infeasible)), marker="x",
                    color="#ba332b", label="Brak rozwiązania dopuszczalnego\n(znacznik przy 0, bez kosztu)")
axes[0].set(xlabel="Minimalna wymagana zawartość białka (%)", ylabel="Minimalny koszt (PLN/kg)",
            title="Koszt ostrzejszej specyfikacji")
axes[0].legend()
share_columns = [f"share_{i}" for i in ingredients.index]
axes[1].stackplot(feasible.protein_target_pct, *[feasible[c] for c in share_columns],
                  labels=ingredients.index, colors=[COLORS[i] for i in ingredients.index], alpha=0.85)
axes[1].set(xlabel="Minimalna wymagana zawartość białka (%)", ylabel="Udział składnika (%)",
            ylim=(0, 100), title="Jak zmienia się optymalna receptura")
axes[1].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()
display(sweep[["protein_target_pct", "status", "cost_pln_per_kg"]].round(3))

### Zadanie 4 — zinterpretuj granicę możliwości

Znajdź odcinek stałego kosztu, odcinek rosnący i przejście do braku rozwiązania dopuszczalnego. Wyjaśnij każdy fragment za pomocą ograniczeń receptury. Koszt może być stały, gdy inna reguła już wymusza więcej białka, niż żądamy. Załamania pojawiają się, gdy zmieniają się ograniczenia aktywne lub wybrane składniki.

Wybierz wymaganie 24%, 28% albo 32% białka. Porównaj dodatkowy koszt **na partię** i **na kg** oraz opisz zmiany receptury. Wyższe wymaganie to decyzja o specyfikacji; optymalizator nie oceni, czy klienci cenią ją wystarczająco wysoko.

## 8. Lokalna wrażliwość: ile kosztuje dodatkowe wymaganie?

CBC podaje dla każdego ograniczenia LP **cenę dualną**, zwaną również ceną cienia. Dla `Protein_kg` jej jednostką jest **PLN na dodatkowy kg wymaganego białka**, przy pozostałych prawych stronach bez zmian. W tym modelu minimalizacji wartość dodatnia oznacza, że zaostrzenie minimum białka lokalnie zwiększa koszt.

Zmiana z 24% na 24,01% dodaje **0,01 kg wymaganego białka** do partii 100 kg: `100 × (0.2401 − 0.24)`. Wzrost o jeden punkt procentowy dodaje 1 kg wymaganego białka.

Ceny dualne opisują lokalne nachylenie ciągłego modelu LP. Mogą się zmieniać przy załamaniach, mogą być niejednoznaczne w optimum zdegenerowanym i nie gwarantują wyniku dużych zmian ani modeli całkowitoliczbowych. Sprawdzamy małą zmianę przez ponowne rozwiązanie zamiast traktować podaną cenę jako globalną obietnicę.

In [ ]:
# Przelicz zmianę wymagania na kg białka przed interpretacją jednostek ceny dualnej LP.
delta_fraction = 0.0001  # 0,01 punktu procentowego
slightly_stricter = solve_blend(min_protein=0.24 + delta_fraction)
if baseline["status"] == slightly_stricter["status"] == "Optimal":
    delta_required_kg = baseline["batch_kg"] * delta_fraction
    local_slope = (slightly_stricter["cost_pln"] - baseline["cost_pln"]) / delta_required_kg
    dual = baseline["model"].constraints["Protein_kg"].pi
    print(f"Cena dualna CBC: {dual:.4f} PLN na kg wymaganego białka")
    print(f"Nachylenie przy małej zmianie: {local_slope:.4f} PLN na kg wymaganego białka")
    print(f"Zaobserwowany wzrost kosztu: {slightly_stricter['cost_pln'] - baseline['cost_pln']:.4f} PLN")
    if dual is not None:
        print(f"Lokalne przewidywanie z ceny dualnej: {dual * delta_required_kg:.4f} PLN")

### Zadanie 5 — sprawdź granice przewidywania

Zmień `delta_fraction` na `0.08` — osiem punktów procentowych, czyli 24% → 32%. Czy początkowa cena cienia nadal przewiduje zaobserwowany wzrost kosztu? Sprawdź, które ograniczenia są aktywne przed zmianą i po niej. Pośrednio wypróbuj `0.04`: większa zmiana może nadal odpowiadać lokalnemu nachyleniu, jeśli ten sam zestaw ograniczeń aktywnych pozostaje właściwy. Zerowa cena dualna może oznaczać, że niewielkie zaostrzenie nic nie kosztuje; aktywność ograniczenia nie zawsze oznacza niezerową cenę dualną.

## 9. Interaktywny eksplorator receptury

Zmieniaj jeden element sterujący naraz, przewiduj skutek, a następnie sprawdzaj status i oba wykresy. Elementy sterujące **ponownie rozwiązują LP**, więc zmiana cen może zmienić recepturę, a nie tylko wyświetlony rachunek.

- **Partia:** 50–250 kg. Przy samych regułach procentowych udziały pozostają takie same, a koszt jest proporcjonalny do masy. Stałe ograniczenie zapasu magazynowego zmieniłoby ten efekt.
- **Białko:** 18–38%. Wysokie ustawienia celowo prowadzą do braku rozwiązania dopuszczalnego.
- **Cena kurczaka ×:** mnożnik stosowany tylko do ceny kurczaka.

Jeżeli widżety są niedostępne, poniższa komórka parametrów wykonuje dokładnie ten sam eksperyment. Suwaki nie są potrzebne do ukończenia ćwiczenia.

In [ ]:
# Eksperyment bez widżetów: zmień te trzy liczby i uruchom komórkę ponownie.
experiment_batch_kg = 100
experiment_min_protein = 0.28
experiment_chicken_cost_multiplier = 1.0

# Argumenty nazwane łączą edytowane parametry z wejściami solvera, zachowując pozostałe wartości domyślne.
experiment = solve_blend(batch_kg=experiment_batch_kg,
                         min_protein=experiment_min_protein,
                         chicken_cost_multiplier=experiment_chicken_cost_multiplier)
plot_blend(experiment)

### Funkcja pomocnicza: odśwież eksperyment interaktywny

`update_explorer(change=None)` jest wywoływana po obserwowanej zmianie wartości suwaka; przyjmuje opcjonalne zdarzenie zmiany, ale go nie odczytuje. Korzysta ze wspólnych elementów sterujących masą partii, białkiem i ceną kurczaka zamiast przyjmować te wartości jako argumenty. Początkowe ustawienia to 100 kg, 24% i mnożnik ceny 1,0. Funkcja obsługi zdarzenia zamienia procent białka na ułamek, tworzy słownik nazwanych parametrów `parameters` i przekazuje go do `solve_blend`.

`plot_blend` sprawdza status przed pokazaniem tabel i wykresów w `explorer_output`. Funkcja obsługi zdarzenia zwraca `None` i zastępuje wcześniejszy wynik zamiast dodawać kolejne rysunki. Ponowne użycie solvera i funkcji rysującej zapewnia zgodność aktualizacji interaktywnych z wariantem bez suwaków powyżej.

In [ ]:
# Utwórz elementy sterujące w czytelnych jednostkach; funkcja obsługi przelicza procenty na ułamki.
batch_slider = widgets.IntSlider(value=100, min=50, max=250, step=10,
                                  description="Partia (kg)", continuous_update=False)
protein_slider = widgets.FloatSlider(value=24, min=18, max=38, step=0.5,
                                     description="Białko (%)", continuous_update=False)
price_slider = widgets.FloatSlider(value=1.0, min=0.5, max=2.5, step=0.1,
                                   description="Kurczak ×", continuous_update=False)
explorer_output = widgets.Output()

def update_explorer(change=None):
    """Zastąp interaktywny widok receptury wynikiem bieżących wartości sterowania.
    
    change=None przyjmuje niewykorzystywane zdarzenie obserwacji ipywidgets.
    Korzysta z globalnych batch_slider (kg), protein_slider (%) i price_slider
    (mnożnik), początkowo 100, 24 i 1.0. Przelicza wartości na argumenty nazwane
    solve_blend, buduje nowy model i używa kontroli statusu w plot_blend.
    Wyświetla wynik w globalnym explorer_output, usuwając poprzednią zawartość;
    zwraca None. Zależy również od clear_output, solve_blend i plot_blend."""
    # 1. Odczytaj wartości sterowania i zamień procent na ułamek.
    parameters = {"batch_kg": batch_slider.value,
                  "min_protein": protein_slider.value / 100,
                  "chicken_cost_multiplier": price_slider.value}
    with explorer_output:  # zachowaj wynik w jednym obszarze wyjścia
        clear_output(wait=True)  # zastąp poprzedni wynik
        # 2. Zbuduj nowy model i rozwiąż go z tymi parametrami.
        result = solve_blend(**parameters)
        # 3. Sprawdź status; 4. pokaż sprawdzoną tabelę i wykresy.
        plot_blend(result)

for control in [batch_slider, protein_slider, price_slider]:
    # .observe wywołuje funkcję obsługi po każdej zmianie wartości elementu sterującego.
    control.observe(update_explorer, names="value")
display(widgets.VBox([batch_slider, protein_slider, price_slider, explorer_output]))
update_explorer()  # pokaż wartości początkowe, zanim ktokolwiek poruszy suwakiem

### Dodaj jeszcze jeden element sterujący

Przepływ danych to **wartość suwaka → przeliczenie parametru → nowy model → sprawdzenie statusu → tabela i wykres**. `observe` łączy element sterujący z funkcją obsługi zdarzenia; samo niczego nie rozwiązuje. `continuous_update=False` czeka na puszczenie suwaka i zapobiega wielu zbędnym rozwiązaniom. Rozpakowanie argumentów `solve_blend(**parameters)` odpowiada osobnemu przekazaniu każdego argumentu nazwanego.

**Zadanie:** dodaj `moisture_slider` z zakresem 20–80% i wartością początkową 60. Dodaj `"max_moisture": moisture_slider.value / 100` do `parameters`, uwzględnij element sterujący w pętli `observe` i w `VBox`, a następnie uruchom całą komórkę panelu. Sprawdź dwa ustawienia i upewnij się, że zmieniają się linia limitu wody oraz status lub receptura, zgodnie z wynikiem. Właśnie został zbudowany nowy element sterujący optymalizacją.

### Zadanie 6 — kontrolowane eksperymenty

Zapisuj krótką tabelę wyników: ustawienia parametrów, status, koszt/kg i jedną obserwację dotyczącą receptury.

1. Porównaj 50 kg i 200 kg przy tych samych wymaganiach i cenach. Sprawdź równe udziały i czterokrotny wzrost łącznego kosztu.
2. Porównaj mnożniki ceny kurczaka `0.5`, `1.0` i `2.0` przy 24% białka. Które składniki zastępują kurczaka, jeśli w ogóle? Wyjaśnij rolę jego minimalnego udziału.
3. Znajdź ustawienie białka bez rozwiązania dopuszczalnego. Wskaż prostą konieczną granicę, która je wyklucza.
4. Opcjonalnie: dodaj do własnego modelu stałe ograniczenie zapasu kurczaka, np. `chicken_kg <= 80`. Wyjaśnij, dlaczego masa partii wpływa teraz na dopuszczalność i udziały składników.

## 10. Twoja rekomendacja

Przekaż zespołowi produkcyjnemu krótką rekomendację zawierającą:

- wybraną masę partii i wymaganie białka, status, dokładną tabelę receptury i koszt/kg;
- wykresy receptury i jakości oraz ograniczenia aktywne;
- jeden dopuszczalny wariant alternatywny i jego dodatkowy koszt;
- jeden scenariusz bez rozwiązania dopuszczalnego, diagnozę i propozycję naprawy;
- eksperyment ze zmianą ceny kurczaka i zdanie o lokalnej wrażliwości;
- jedno ograniczenie modelu: uproszczone dane, ilości ciągłe, stałe ceny jednostkowe lub brak ograniczeń produkcji i zapasów.

**Przed przekazaniem receptury:** sprawdź bilans masy, każde wymaganie jakościowe i każdą granicę udziału. Samo podanie wartości funkcji celu bez sprawdzenia statusu i jednostek nie wystarcza.

Laboratorium 3 przechodzi od **składu mieszanki** do **przepływu towarów i rozkroju surowca**. Laboratorium 6 połączy mieszanie z niepewnymi cenami i działaniem zrobotyzowanej komórki produkcyjnej.